# Goal logistic regression: tuning and test evaluation

The model estimates whether the **same team scores a goal before the inferred
possession ends**, using only states before the first explicit shot. It uses the
five current-state features and the audited labels from notebook 06.

The experiment fits candidates on **2023–2024**, selects L2 regularisation using
**2025**, refits the complete pipeline on **2023–2025**, then scores **2026**.
Each possession contributes total weight one across its prefixes. The reporting
threshold stays at **0.5** and is not tuned.

This notebook reads saved results. Run `python -m afl_opposition.goal_tuning`
after preparing all four seasons to generate them. Rerunning the notebook does
not train, tune or rescore the test data. Saved outputs are included. Run cells
in order using the project kernel.
See [the experiment specification](../docs/tuned-goal-logistic.md).

## 1. Experiment records

Experiment paths are resolved relative to the repository. The goal experiment
has its own artifact folder, separate from the earlier shot results. `selection`
describes the training-only winner, `refit` describes the combined-data model,
and `test` describes its subsequent evaluation.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data/sources.json").is_file() and (p / "pyproject.toml").is_file()), None,
)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
experiment = project_root / "artifacts/logistic-goal-tuned-2026"
if not (experiment / "test/summary.json").is_file():
    raise RuntimeError("Prepare the four seasons, then run python -m afl_opposition.goal_tuning.")
selection = json.loads((experiment / "selection/summary.json").read_text())
refit = json.loads((experiment / "refit.json").read_text())
test = json.loads((experiment / "test/summary.json").read_text())
trials = pd.read_parquet(experiment / "lambda_search.parquet")
assert all(record["prediction_target"] == "goal" for record in [selection, refit, test])
print("Experiment:", experiment.relative_to(project_root).as_posix())
print("Features:", selection["feature_columns"])
print("Encoded feature count:", selection["encoded_feature_count"])

Experiment: artifacts/logistic-goal-tuned-2026
Features: ['action_type', 'x_norm', 'y_norm', 'gap_seconds', 'elapsed_seconds']
Encoded feature count: 16


## 2. Training objective and selection rule

For prefix weight $s_i$, $S=\sum_i s_i$, coefficients $\beta$ and unpenalised
intercept $b$, the training objective is

$$p_i=\sigma(\beta^T x_i+b),\qquad
J=\frac{1}{S}\sum_i s_i\operatorname{BCE}(y_i,p_i)+\lambda\|\beta\|_2^2.$$

The encoder and scaler are fitted on 2023–2024 only. Each candidate receives a
fresh logistic classifier. The selection criterion is **weighted 2025 BCE without
the penalty term**. AP and Brier score are diagnostics, not selection criteria.

The shared shot/goal procedure tests zero and powers of ten from $10^{-8}$ to
$10^1$, checks a boundary when needed, then refines around the best positive
candidate at quarter-decade intervals. Exact loss ties favour larger lambda.
Validation determines these choices; test outcomes do not.

In [2]:
display(pd.DataFrame({
    "setting": ["Training", "Validation", "Selection criterion", "Solver", "Tolerance", "Max iterations", "Threshold"],
    "value": [str(selection["training_seasons"]), str(selection["validation_season"]),
              selection["selection_metric"], selection["solver"], str(selection["tol"]),
              str(selection["max_iter"]), str(selection["decision_threshold"])],
}))
print("Coarse lambda grid:", selection["coarse_lambdas"])
assert selection["test_used_for_selection"] is False

,setting,value
0,Training,"[2023, 2024]"
1,Validation,2025
2,Selection criterion,minimum possession-weighted 2025 binary cross-...
3,Solver,lbfgs
4,Tolerance,1e-08
5,Max iterations,2000
6,Threshold,0.5


Coarse lambda grid: [0.0, 1e-08, 1e-07, 1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]


## 3. Regularisation search results

The complete trial table is sorted by lambda, with the selected row identified. Smaller
weighted validation BCE is better. The unregularised candidate checks whether
the data support adding a positive penalty. Differences between nearby values
can be small; the selected minimum is not evidence of a uniquely optimal value.

The validation score has been used for tuning. It is not the final evaluation.

In [3]:
display(trials.sort_values("lambda_l2").reset_index(drop=True).style.format({
    "lambda_l2": "{:.8g}", "C": "{:.8g}", "training_log_loss": "{:.9f}",
    "validation_log_loss": "{:.9f}", "validation_average_precision": "{:.6f}",
    "validation_brier_score": "{:.6f}",
}))
winner = trials.loc[trials.selected].iloc[0]
assert trials.selected.sum() == 1
assert winner.validation_log_loss == trials.validation_log_loss.min()
assert winner.lambda_l2 == selection["lambda_l2"] == refit["lambda_l2"]
print("Selected lambda:", refit["lambda_l2"])
print("Selected validation BCE:", winner.validation_log_loss)

,lambda_l2,C,stage,training_weight_sum,training_log_loss,validation_log_loss,validation_average_precision,validation_brier_score,iterations,selected
0,0,inf,coarse,103078.000000,0.280017776,0.287695954,0.274692,0.082140,122,False
1,1e-08,485.06956,coarse,103078.000000,0.280017776,0.287695952,0.274692,0.082140,311,False
2,1e-07,48.506956,coarse,103078.000000,0.280017776,0.287695926,0.274692,0.082140,217,False
3,1e-06,4.8506956,coarse,103078.000000,0.280017789,0.287695708,0.274690,0.082140,204,False
4,1.7782794e-06,2.7277466,refined,103078.000000,0.280017817,0.287695529,0.274687,0.082140,192,False
5,3.1622777e-06,1.5339246,refined,103078.000000,0.280017901,0.287695236,0.274686,0.082140,287,False
6,5.6234133e-06,0.86258921,refined,103078.000000,0.280018153,0.287694796,0.274681,0.082140,169,False
7,1e-05,0.48506956,coarse,103078.000000,0.280018885,0.287694261,0.274668,0.082141,172,False
8,1.7782794e-05,0.27277466,refined,103078.000000,0.280020938,0.287694015,0.274644,0.082141,205,True
9,3.1622777e-05,0.15339246,refined,103078.000000,0.280026481,0.287695606,0.274598,0.082143,168,False


Selected lambda: 1.778279410038923e-05
Selected validation BCE: 0.287694015112963


## 4. Refitting with the selected regularisation

A fresh encoder, scaler and classifier are fitted on all eligible 2023–2025
examples. The selected coefficient of $\|\beta\|^2$ remains fixed. Scikit-learn's
equivalent inverse-regularisation parameter is $C=1/(2\lambda S)$, so `C` changes
when the fitting weight sum changes. For lambda zero, `C` is infinite.

The constant benchmark is also learned from the combined development data.
It is not set to the test prevalence. Possession counts differ from prefix counts
because one possession can contribute multiple correlated predictions.

In [4]:
display(pd.DataFrame([
    {"stage": "Candidate fitting", "seasons": "2023–2024", "weight_sum": selection["training_weight_sum"],
     "lambda": selection["lambda_l2"], "C": selection["C"]},
    {"stage": "Final refit", "seasons": "2023–2025", "weight_sum": refit["training_weight_sum"],
     "lambda": refit["lambda_l2"], "C": refit["C"]},
]))
if refit["lambda_l2"] > 0:
    assert abs(1 / (2 * refit["C"] * refit["training_weight_sum"]) - refit["lambda_l2"]) < 1e-12
print("Refit matches:", refit["refit_matches"])
print("Refit possessions:", refit["refit_possessions"])
print("Refit prefixes:", refit["refit_examples"])

,stage,seasons,weight_sum,lambda,C
0,Candidate fitting,2023–2024,103078.0,0.000018,0.272775
1,Final refit,2023–2025,153822.0,0.000018,0.182790


Refit matches: 647
Refit possessions: 153822
Refit prefixes: 844775


## 5. Frozen-model verification and test-label coverage

The model and refit record were saved **before the 2026 loader was called**.
Checksums link the model and trial search to the test report. The production run
also verified that the in-memory model and implementation did not change during
test evaluation.

Goal labels use the same rules as notebook 06. The tables distinguish quarantined
goal annotations, verified attribution, and possession eligibility. A verified
goal may still be excluded by the pre-shot or preprocessing policy.

The 2026 season was previously used for the shot benchmark. This is a separately
selected goal baseline on the same season, not a claim of a newly untouched
holdout. No goal-specific tuning uses 2026.

In [5]:
def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

assert file_hash(experiment / "refit_model.pkl") == refit["model_sha256"]
assert file_hash(experiment / "refit.json") == test["refit_sha256"]
assert file_hash(experiment / "lambda_search.parquet") == refit["lambda_search_sha256"]
assert refit["test_loaded_after_refit"] is True
print("Model frozen (UTC):", refit["frozen_at_utc"])
print("Evaluation completed (UTC):", test["test_evaluated_at_utc"])
labels = json.loads((experiment / "test_labels/summary.json").read_text())
for name, digest in labels["artifact_sha256"].items():
    assert file_hash(experiment / "test_labels" / f"{name}.parquet") == digest
display(pd.read_parquet(experiment / "test_labels/coverage.parquet"))
display(pd.DataFrame.from_dict(labels["status_counts"], orient="index", columns=["possessions"]))
display(pd.DataFrame.from_dict(labels["goal_annotation_status_counts"], orient="index", columns=["goal_annotations"]))
print("Additional goal-attribution exclusions:", labels["additional_goal_exclusions"])

Model frozen (UTC): 2026-10-11T14:02:56.750136+00:00
Evaluation completed (UTC): 2026-10-11T14:03:29.814353+00:00


,season,matches,total_possessions,eligible_possessions,goal_possessions,no_goal_possessions,excluded_possessions,prefix_examples,goal_possession_rate,raw_goals,quarantined_goals,verified_goals,multi_shot_goal_possessions
0,2026,218,54892,51234,5421,45813,3658,290909,0.105809,5644,16,5628,88


,possessions
eligible,51234
preprocessing_review,3629
no_pre_shot_action,29


,goal_annotations
verified,5628
quarantined_quarter,16


Additional goal-attribution exclusions: 0


## 6. Test probability metrics and constant benchmark

All metrics in this table use possession weights. Lower BCE and Brier score are
better; higher AP is better. Evaluation BCE excludes the training L2 penalty.
The constant predictor's test AP equals the weighted goal prevalence because all
its predictions are tied. Its probability itself was fitted on 2023–2025.

The goal and shot tasks have different prevalences and labels. Their raw AP
values cannot establish which model is better without accounting for that
different task. The goal constant benchmark provides the relevant reference.

In [6]:
metrics = pd.read_parquet(experiment / "test/metrics.parquet")
assert set(metrics.split) == {"test"}
display(metrics[["model", "examples", "possessions", "observed_goal_rate",
                 "log_loss", "brier_score", "average_precision"]].round(6))
print("Constant goal probability learned on 2023–2025:", test["constant_probability"])

,model,examples,possessions,observed_goal_rate,log_loss,brier_score,average_precision
0,constant,290909,51234,0.105809,0.337951,0.094666,0.105809
1,logistic,290909,51234,0.105809,0.297569,0.085786,0.284618


Constant goal probability learned on 2023–2025: 0.09855547320929386


## 7. Confusion matrices at the fixed threshold

The positive class is a subsequent goal in the same possession. Rows are actual
outcomes and columns are predictions. Raw counts refer to action states, not
unique possessions. Weighted cells sum fractions of possession weights; an
individual possession can contribute to more than one prediction column.

Recall and false negatives quantify missed goal-producing states; precision
and false positives describe positive predictions. High accuracy alone can
reflect the majority no-goal class. The threshold remains
0.5 even if the resulting recall is low; test results do not select a threshold.

In [7]:
confusion = pd.read_parquet(experiment / "test/confusion.parquet")
logistic = confusion.loc[confusion.model.eq("logistic")]
for row in logistic.itertuples(index=False):
    print(row.weighting, "| threshold:", row.threshold)
    matrix = pd.DataFrame([[row.tn, row.fp], [row.fn, row.tp]],
        index=["Actual no goal", "Actual goal"], columns=["Predict no goal", "Predict goal"])
    display(matrix.astype(int) if row.weighting == "unweighted" else matrix.round(2))
rates = ["accuracy", "precision", "recall", "specificity", "f1", "balanced_accuracy"]
print("Rates in percent:")
display((logistic.set_index("weighting")[rates].T * 100).round(2))
display(confusion.loc[confusion.weighting.eq("possession_weighted"), ["model", "threshold", *rates]].round(4))

unweighted | threshold: 0.5


,Predict no goal,Predict goal
Actual no goal,248910,615
Actual goal,40610,774


possession_weighted | threshold: 0.5


,Predict no goal,Predict goal
Actual no goal,45765.58,47.42
Actual goal,5352.83,68.17


Rates in percent:


weighting,unweighted,possession_weighted
accuracy,85.83,89.46
precision,55.72,58.98
recall,1.87,1.26
specificity,99.75,99.90
f1,3.62,2.46
balanced_accuracy,50.81,50.58


,model,threshold,accuracy,precision,recall,specificity,f1,balanced_accuracy
1,constant,0.5,0.8942,NaN,0.0000,1.000,0.0000,0.5000
3,logistic,0.5,0.8946,0.5898,0.0126,0.999,0.0246,0.5058


## 8. Calibration and evaluation artifacts

Fixed probability bins compare mean predictions with observed goal rates. This
is descriptive evaluation, not calibration fitted on test outcomes. The output
folders include predictions and audits for later review.

These results concern the eligible archived sample. Inferred possession
boundaries, source completeness, excluded events and correlated prefixes limit
the interpretation. The model estimates a probability from recorded features;
it does not establish causal action value or live-match reliability.

In [8]:
calibration = pd.read_parquet(experiment / "test/calibration.parquet")
display(calibration.loc[calibration.model.eq("logistic"), [
    "bin_lower", "bin_upper", "examples", "weight_sum", "mean_prediction", "observed_goal_rate",
]].round(4))
print("Regularisation search:", "artifacts/logistic-goal-tuned-2026/lambda_search.csv")
print("Frozen model:", "artifacts/logistic-goal-tuned-2026/refit_model.pkl")
print("Test evaluation:", "artifacts/logistic-goal-tuned-2026/test/")
print("Test goal-label audit:", "artifacts/logistic-goal-tuned-2026/test_labels/")

,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_goal_rate
1,0.0,0.1,181250,32728.9640,0.0461,0.0525
2,0.1,0.2,68629,11556.3331,0.1425,0.1485
3,0.2,0.3,26091,4590.5983,0.2410,0.2154
4,0.3,0.4,9578,1587.8849,0.3434,0.3722
5,0.4,0.5,3972,654.6310,0.4398,0.5188
6,0.5,0.6,1160,100.7573,0.5325,0.5851
7,0.6,0.7,199,13.2907,0.6362,0.6333
8,0.7,0.8,26,1.1957,0.7380,0.4550
9,0.8,0.9,4,0.3450,0.8558,0.7360


Regularisation search: artifacts/logistic-goal-tuned-2026/lambda_search.csv
Frozen model: artifacts/logistic-goal-tuned-2026/refit_model.pkl
Test evaluation: artifacts/logistic-goal-tuned-2026/test/
Test goal-label audit: artifacts/logistic-goal-tuned-2026/test_labels/
